# RAG
Retrieval-Augmented Generation (RAG) system is an AI architecture that enhances large language models (LLMs) by fetching specific, external data before generating an answer

Here, we are goin to use first the Kerword search method.

Keyword search = same words <br>
Vector search = same meaning <br>
Agent = LLM controls the search process <br>

RAG is the full pipeline: <br>
question → search → context → LLM answer

The “search” part can be different.

In [1]:
# load_dotenv() returns if it found and loaded a .env file
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
# Load the open AI client
from openai import OpenAI
openai_client = OpenAI()

Plain LLMs lack our data

In [3]:
# First, let's define a function to talk to the LLM
# openai_client.responses.create(...) = object.attribute.method()
# response.output_text = You're reading a value stored inside the response object.
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [4]:
# Let's test it!
llm("Write a short poem about a lonely computer.")

'A lonely computer hums at night,  \nIn a blue-lit room with no one in sight.  \nIt waits for a click, a voice, a sign,  \nThrough silent hours and glowing time.  \n\nIts screen stays bright, its fan goes low,  \nIt dreams of words it’ll never know.  \nYet in the dark, it softly gleams—  \nA quiet heart of wires and dreams.'

In [5]:
# Ask it a course-specific question
question = "I just discovered the course.Can I join now?"
answer = llm(question)
print(answer)

Yes — you can usually join after a course has already started, but it depends on the course’s policy and whether there are still spots available.

If you want, I can help you write a quick message like:

> Hi, I just discovered the course and I’m very interested in joining. Is it still possible to enroll at this stage?

If you’d like, I can also help you make it sound more formal or more friendly.


The LLM gives a generic answer, since our courses are not in the training data.

Adding context manually

More context can fix this. The FAQ website has questions and answers about our courses.

In [6]:
#Copy some of that content into the prompt
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


In [7]:
#Build a prompt that includes both the question and the context
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
#Instead of sending the raw question to the LLM, we send this prompt
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.

The pieces are search, the prompt, and the LLM:<br>
search <br>
prompt <br>
LLM <br>

In code, it looks like this:

In [9]:
#This code can't run now
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

# 04 The Course FAQ Dataset

In [10]:
# Load the requests library so we can call web APIs.
import requests

In [11]:
# Create a variable called docs_url and set it to the URL of the courses documentation.
docs_url = "https://datatalks.club/faq/json/courses.json"

In [12]:
# Send a GET request to the URL.
# requests is a module/library and get() is a function inside that module.
response = requests.get(docs_url)

In [13]:
# response = object
# json() = method of the Response object
# json() parses that JSON text and converts it into Python objects.
# If JSON Object{}
courses_raw = response.json()

In [14]:
# let's check what we got!
courses_raw 

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 84},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

Playing around with dictionaries 

In [15]:
# data types
type(courses_raw) # list

list

In [16]:
# see data type of the first element in the list
type(courses_raw[0]) # dict

dict

In [17]:
# Let's look at the first course in the list
course_1 = courses_raw[0]
course_1

{'course': 'data-engineering-zoomcamp',
 'course_name': 'Data Engineering Zoomcamp',
 'path': '/json/data-engineering-zoomcamp.json',
 'questions_count': 404}

In [18]:
# see keys of the course_1 dictionary
course_1.keys()

dict_keys(['course', 'course_name', 'path', 'questions_count'])

In [19]:
# see values of the course_1 dictionary
course_1.values()

dict_values(['data-engineering-zoomcamp', 'Data Engineering Zoomcamp', '/json/data-engineering-zoomcamp.json', 404])

In [20]:
# see the value of the 'course' key in the course_1 dictionary
course_1['course']

'data-engineering-zoomcamp'

In [21]:
# see the value of the 'course' key in the course_1 dictionary using the get() method
course_1.get('course')

'data-engineering-zoomcamp'

Back to our exercise. <br>
Let's fetch all the FAQ documents from all courses:


In [22]:
# create documents list to store the course URLs
documents = []
url_prefix = "https://datatalks.club/faq"

# go through each course in the courses_raw list and print the course URL
for course in courses_raw:
    course_url = f"{url_prefix}{course["path"]}"
    documents.append(course_url)
    print(course_url)

https://datatalks.club/faq/json/data-engineering-zoomcamp.json
https://datatalks.club/faq/json/stock-markets-analytics-zoomcamp.json
https://datatalks.club/faq/json/ai-dev-tools-zoomcamp.json
https://datatalks.club/faq/json/llm-zoomcamp.json
https://datatalks.club/faq/json/mlops-zoomcamp.json
https://datatalks.club/faq/json/machine-learning-zoomcamp.json


In [23]:
documents

['https://datatalks.club/faq/json/data-engineering-zoomcamp.json',
 'https://datatalks.club/faq/json/stock-markets-analytics-zoomcamp.json',
 'https://datatalks.club/faq/json/ai-dev-tools-zoomcamp.json',
 'https://datatalks.club/faq/json/llm-zoomcamp.json',
 'https://datatalks.club/faq/json/mlops-zoomcamp.json',
 'https://datatalks.club/faq/json/machine-learning-zoomcamp.json']

In [24]:
# Go to the URL and download its contents.
# We will use the first URL in the documents list as an example.
# Go to the URL and download its contents.
course_response = requests.get(documents[0])
course_response

<Response [200]>

In [25]:
# checks that status code and crashes if it's not successful. 
course_response.raise_for_status()

In [26]:
# Convert the JSON returned by the server into Python objects.
course_data = course_response.json()
course_data

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

we are going to use extend() and not append because extend says : <br>
"Take all the FAQ entries from this course and add them one by one to the master list."

In [27]:
# final query to collect all questions and answers from the course page
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1349

In [28]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [29]:
# using slicing to see the first 5 documents
documents[:3]

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

In the RAG pipeline, this dataset is our knowledge base

# Search

## Indexing with minsearch

In [30]:
# Import the Index class from the minsearch library.
from minsearch import Index

# Create an Index object.
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

In [31]:
index

minsearch: <br>
* Loops through all documents.
* Reads the text fields.
* Tokenizes them into words.
* Builds the search index.

Conceptually: <br>
"join"      → doc 17, doc 48  <br>
"homework"  → doc 3, doc 92  <br> 
"certificate" → doc 10, doc 51  <br>

In [32]:
# Build the index using the documents.
index.fit(documents)


## Trying a search

boost_dict={"question": 2.0, "section": 0.5}

This controls importance of fields.

Meaning: <br>
question field = more important ×2 <br>
section field = less important ×0.5 <br>
answer field = normal importance ×1 <br>

In [33]:
# Let's try a search with the question we used before
question = "I just discovered the course. Can I join now?"

#index.search(...) --> Searches inside the index you built before.
search_results = index.search(
    question,
    # This controls importance of fields.
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results


[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [34]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

## Wrapping it in a function
Let's wrap the search in a search function - the first component of our RAG pipeline

In [35]:
# By default it searches the LLM Zoomcamp FAQ
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [36]:
question_1 = "the course it's free?"

In [37]:
search_results = search(question_1)

In [38]:
search_results

[{'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit answers through the course platform before the deadline.\n\nHomework is similar to the lesson flow, but uses a different dataset or slightly different task.'},
 {'id': '0d74a3616f

# Building the Prompt

In [39]:
# The instructions tell the LLM its role and how to answer
# This is what grounds the answer in our data and reduces hallucinations.

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [40]:
# The user prompt template has placeholders for the question and the context
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [41]:
# Building the context
# The context is a formatted string with all the search results
# We turn a list of dictionaries into one string
def build_context(search_results):
    # creates a string
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
        
    #.strip() removes whitespace from the beginning and end of a string.
    # lines.append("") adds an empty line after every document.
    return "\n".join(lines)

"\n".join(lines)

means:

"Take all items in the list and combine them into one string, putting a newline (\n) between each item. Since LLM get strings, not lists"

In [42]:
context = build_context(search_results)
context

'General Course-Related Questions\nQ: How should I start the course and follow the weekly workflow?\nA: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit answers through the course platform before the deadline.\n\nHomework is similar to the lesson flow, but uses a different dataset or slightly different task.\n\nModule 1: Agentic RAG\nQ: Any free models with tool use support?\nA: Several Groq models offer tool use, s

In [43]:
# Building the prompt
# Now we combine the question with the context into the user prompt
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [44]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: How should I start the course and follow the weekly workflow?
A: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

A typical workflow is:

1. Watch the lesson videos.
2. Work through the lesson notebooks/code.
3. Read the homework instructions on GitHub.
4. Submit answers through the course platform before the deadline.

Homework is similar to the lesson flow, but uses a different dataset or slightly different task.

Module 1: Agentic RAG
Q: Any free models with tool use s

# The LLM
The last component of our RAG pipeline is the LLM. It takes the prompt we built and generates an answer.

In [45]:
# send prompt to the LLM
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

The response is a Pydantic object. The answer is in response.output - a list of output items.

In [46]:
response

Response(id='resp_0831c36679098bc1006a3648ec167c8192b21b79a218875852', created_at=1781942508.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0831c36679098bc1006a3648eca7088192a715d230ecb78214', content=[ResponseOutputText(annotations=[], text='Yes — you can join now.\n\nYou don’t need a confirmation email; if you’ve registered, you’re already accepted, and even registration isn’t required to start learning. The videos and GitHub materials are available now, and you can follow the course at your own pace while the homework submission form is open.\n\nIf you want, I can also point you to the best place to start.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1781942509.0, conversation=None, max_

In [47]:
# The first output item is the message
response.output[0]

ResponseOutputMessage(id='msg_0831c36679098bc1006a3648eca7088192a715d230ecb78214', content=[ResponseOutputText(annotations=[], text='Yes — you can join now.\n\nYou don’t need a confirmation email; if you’ve registered, you’re already accepted, and even registration isn’t required to start learning. The videos and GitHub materials are available now, and you can follow the course at your own pace while the homework submission form is open.\n\nIf you want, I can also point you to the best place to start.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [48]:
# The message has a content list, and the text is in the first item:
response.output[0].content[0].text

'Yes — you can join now.\n\nYou don’t need a confirmation email; if you’ve registered, you’re already accepted, and even registration isn’t required to start learning. The videos and GitHub materials are available now, and you can follow the course at your own pace while the homework submission form is open.\n\nIf you want, I can also point you to the best place to start.'

In [49]:
# The shortcut spares us all of it
# Same result, less code.
response.output_text

'Yes — you can join now.\n\nYou don’t need a confirmation email; if you’ve registered, you’re already accepted, and even registration isn’t required to start learning. The videos and GitHub materials are available now, and you can follow the course at your own pace while the homework submission form is open.\n\nIf you want, I can also point you to the best place to start.'

In [50]:
# The usage counts tell you how many tokens the request consumed
response.usage

ResponseUsage(input_tokens=556, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=83, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=639)

## Calculating the price
You can use different models.

In this course we'll use gpt-5.4-mini:

Input: $0.75 per million tokens <br>
Output: $4.50 per million tokens <br>
Let's calculate the cost of the request we just made <br>

In [51]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0007905

## Message history
We won't build a multi-turn chat here. But we still use this message format to separate our instructions from the user prompt.

We send two messages: <br>
developer - system-level instructions (how the LLM should behave) <br>
user - the actual prompt with the question and context <br>

In [52]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [53]:
message_history

[{'role': 'developer',
  'content': '\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n'},
 {'role': 'user',
  'content': 'Question:\nI just discovered the course. Can I join now?\n\nContext:\nGeneral Course-Related Questions\nQ: How should I start the course and follow the weekly workflow?\nA: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. 

## The LLM function
We can now put this together into an updated llm function.




In [54]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [55]:
response = llm(INSTRUCTIONS, prompt)

In [56]:
response

'Yes — you can start whenever you want. The videos and GitHub materials are available, and you can begin learning and submitting homework while the form is open.'

## Full RAG

In [57]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [60]:
answer = rag("I need to pay for the course?")
print(answer)

No, you don’t have to pay for the course. You can complete the homeworks using free or low-cost alternatives instead of paying for OpenAI API.
